In [1]:
import json
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.metrics import confusion_matrix, f1_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
processed_dir = PROJECT_ROOT / "data" / "processed"
models_dir = PROJECT_ROOT / "models"
results_dir = PROJECT_ROOT / "results"

with open(processed_dir / "split_config.json", "r", encoding="utf-8") as f:
    split_cfg = json.load(f)

RANDOM_STATE = split_cfg["random_state"]
URUTAN_KELAS = split_cfg["urutan_kelas"]
SORTED_KELAS = sorted(URUTAN_KELAS)
SELECTED_FEATURES = split_cfg["selected_features"]
ATTACK_CLASSES = ["DoS", "Probe", "R2L", "U2R"]

# Definisi ulang class wrapper agar joblib.load('xgboost.joblib') berjalan lancar
class StringXGBClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.random_state = random_state

    def fit(self, X, y):
        self.le_ = LabelEncoder()
        self.le_.fit(URUTAN_KELAS)
        y_enc = self.le_.transform(y)
        self.classes_ = self.le_.classes_
        self.model_ = XGBClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            learning_rate=self.learning_rate,
            random_state=self.random_state,
            eval_metric="mlogloss",
            n_jobs=-1,
        )
        self.model_.fit(X, y_enc)
        return self

    def predict(self, X):
        preds = self.model_.predict(X)
        return self.le_.inverse_transform(preds)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

    @property
    def feature_importances_(self):
        return self.model_.feature_importances_

train_df = pd.read_csv(processed_dir / "train_split_80_20.csv")
test_df = pd.read_csv(processed_dir / "test_split_80_20.csv")
X_train, y_train = train_df[SELECTED_FEATURES], train_df["kategori"]
X_test, y_test = test_df[SELECTED_FEATURES], test_df["kategori"]

# Daftar 5 model (3 dari A3 + 2 dari A2)
MODEL_SPECS = [
    ("extra_trees", "Extra Trees", "Ensemble Pohon"),
    ("xgboost", "XGBoost", "Boosting"),
    ("decision_tree", "Decision Tree", "Pohon Tunggal"),
    ("svm_rbf", "SVM (RBF)", "Kernel / Margin"),
    ("mlp", "MLP", "Jaringan Saraf"),
]

# 1. Jalankan 5-Fold Stratified CV untuk Mean & Std (Macro-F1 & Recall U2R)
print("Menjalankan 5-Fold Stratified CV untuk kelima model...")
cv_5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []

for key, nama, keluarga in MODEL_SPECS:
    print(f"-> Evaluasi CV & Test: {nama}")
    pipe = joblib.load(models_dir / f"{key}.joblib")
    with open(results_dir / f"metrics_{key}.json", "r", encoding="utf-8") as f:
        m_json = json.load(f)

    fold_macro_f1 = []
    fold_recall_u2r = []
    fold_recall_r2l = []
    for tr_idx, val_idx in cv_5.split(X_train, y_train):
        m_clone = clone(pipe)
        m_clone.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        preds = m_clone.predict(X_train.iloc[val_idx])
        fold_macro_f1.append(
            f1_score(y_train.iloc[val_idx], preds, labels=URUTAN_KELAS, average="macro", zero_division=0)
        )
        rec_per_cls = recall_score(
            y_train.iloc[val_idx], preds, labels=URUTAN_KELAS, average=None, zero_division=0
        )
        fold_recall_r2l.append(rec_per_cls[URUTAN_KELAS.index("R2L")])
        fold_recall_u2r.append(rec_per_cls[URUTAN_KELAS.index("U2R")])

    t0 = time.time()
    y_pred_test = pipe.predict(X_test)
    pred_time = round(time.time() - t0, 4)

    idx_sorted = [list(pipe.classes_).index(c) for c in SORTED_KELAS]
    y_prob_test_sorted = pipe.predict_proba(X_test)[:, idx_sorted]
    roc_ovr = float(
        roc_auc_score(y_test, y_prob_test_sorted, multi_class="ovr", average="macro", labels=SORTED_KELAS)
    )

    rep_test = m_json["test"]["classification_report"]
    attack_rec_test = float(np.mean([rep_test[c]["recall"] for c in ATTACK_CLASSES]))
    overfit_gap = float(m_json["train"]["macro_f1"] - m_json["test"]["macro_f1"])

    rows.append({
        "model_key": key,
        "model_name": nama,
        "keluarga": keluarga,
        "test_accuracy": round(m_json["test"]["accuracy"], 4),
        "test_macro_f1": round(m_json["test"]["macro_f1"], 4),
        "test_roc_auc": round(roc_ovr, 4),
        "attack_recall_macro": round(attack_rec_test, 4),
        "recall_r2l_test": round(rep_test["R2L"]["recall"], 4),
        "recall_u2r_test": round(rep_test["U2R"]["recall"], 4),
        "cv_macro_f1_mean": round(float(np.mean(fold_macro_f1)), 4),
        "cv_macro_f1_std": round(float(np.std(fold_macro_f1)), 4),
        "cv_recall_u2r_mean": round(float(np.mean(fold_recall_u2r)), 4),
        "cv_recall_u2r_std": round(float(np.std(fold_recall_u2r)), 4),
        "overfit_gap_f1": round(overfit_gap, 4),
        "train_time_s": round(float(m_json["train_time_s"]), 2),
        "predict_time_s": pred_time,
    })

df_comp = pd.DataFrame(rows)

# 2. Perhitungan Skor Berbobot (Kriteria A.8: Macro-F1 40%, Recall Serangan 30%, Overfitting 15%, Waktu 15%)
def minmax_benefit(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-12)

def minmax_cost(s):
    return (s.max() - s) / (s.max() - s.min() + 1e-12)

df_comp["skor_f1"] = minmax_benefit(df_comp["cv_macro_f1_mean"])
df_comp["skor_recall_serangan"] = minmax_benefit(df_comp["attack_recall_macro"])
df_comp["skor_anti_overfit"] = minmax_cost(df_comp["overfit_gap_f1"].clip(lower=0))
df_comp["skor_waktu"] = minmax_cost(df_comp["train_time_s"])

df_comp["skor_total"] = (
    0.40 * df_comp["skor_f1"]
    + 0.30 * df_comp["skor_recall_serangan"]
    + 0.15 * df_comp["skor_anti_overfit"]
    + 0.15 * df_comp["skor_waktu"]
).round(4)

df_comp = df_comp.sort_values("skor_total", ascending=False).reset_index(drop=True)
df_comp["ranking"] = df_comp.index + 1

df_comp.to_csv(results_dir / "ranking.csv", index=False)
ranking_json = {
    "bobot_kriteria": {
        "cv_macro_f1_mean": 0.40,
        "attack_recall_macro": 0.30,
        "overfit_gap_f1": 0.15,
        "train_time_s": 0.15,
    },
    "ranking": df_comp.to_dict(orient="records"),
}
with open(results_dir / "ranking.json", "w", encoding="utf-8") as f:
    json.dump(ranking_json, f, indent=2)

print("\n=== TABEL PERBANDINGAN & RANKING 5 MODEL ===")
display(df_comp[[
    "ranking", "model_name", "keluarga", "test_accuracy", "test_macro_f1",
    "cv_macro_f1_mean", "cv_macro_f1_std", "attack_recall_macro",
    "recall_u2r_test", "overfit_gap_f1", "train_time_s", "skor_total"
]])

# 3. Ekstraksi 10 Fitur Terpenting dari Extra Trees dan XGBoost
def get_top_features(pipe, model_label, top_n=10):
    prep = pipe.named_steps["prep"]
    clf = pipe.named_steps["clf"]
    feat_names = prep.get_feature_names_out()
    importances = clf.feature_importances_
    df_fi = pd.DataFrame({
        "model": model_label,
        "feature": feat_names,
        "importance": importances,
    }).sort_values("importance", ascending=False).head(top_n)
    df_fi["rank"] = range(1, len(df_fi) + 1)
    return df_fi

et_pipe = joblib.load(models_dir / "extra_trees.joblib")
xgb_pipe = joblib.load(models_dir / "xgboost.joblib")

df_fi_all = pd.concat([
    get_top_features(et_pipe, "Extra Trees", 10),
    get_top_features(xgb_pipe, "XGBoost", 10),
], ignore_index=True)

df_fi_all.to_csv(results_dir / "feature_importance.csv", index=False)
print("\n=== TOP 10 FITUR TERPENTING (EXTRA TREES & XGBOOST) ===")
display(df_fi_all)

# 4. Analisis Kesalahan Khusus U2R dan R2L pada Model Peringkat 1
best_key = df_comp.loc[0, "model_key"]
best_name = df_comp.loc[0, "model_name"]
best_pipe = joblib.load(models_dir / f"{best_key}.joblib")
y_pred_best = best_pipe.predict(X_test)
cm_best = pd.DataFrame(
    confusion_matrix(y_test, y_pred_best, labels=URUTAN_KELAS),
    index=[f"Aktual_{c}" for c in URUTAN_KELAS],
    columns=[f"Pred_{c}" for c in URUTAN_KELAS],
)
print(f"\n=== MATRIKS KEKELIRUAN DETAIL R2L & U2R — {best_name} ===")
display(cm_best.loc[["Aktual_R2L", "Aktual_U2R"]])

Menjalankan 5-Fold Stratified CV untuk kelima model...
-> Evaluasi CV & Test: Extra Trees
-> Evaluasi CV & Test: XGBoost
-> Evaluasi CV & Test: Decision Tree
-> Evaluasi CV & Test: SVM (RBF)
-> Evaluasi CV & Test: MLP

=== TABEL PERBANDINGAN & RANKING 5 MODEL ===


,ranking,model_name,keluarga,test_accuracy,test_macro_f1,cv_macro_f1_mean,cv_macro_f1_std,attack_recall_macro,recall_u2r_test,overfit_gap_f1,train_time_s,skor_total
0,1,XGBoost,Boosting,0.9926,0.9646,0.9527,0.0087,0.9539,0.8333,0.0354,39.02,0.9319
1,2,Extra Trees,Ensemble Pohon,0.9924,0.9507,0.9524,0.0136,0.9334,0.7500,0.0493,44.85,0.7066
2,3,SVM (RBF),Kernel / Margin,0.9872,0.9459,0.9329,0.0167,0.9292,0.7500,0.0303,140.36,0.3647
3,4,MLP,Jaringan Saraf,0.9826,0.9245,0.9238,0.0183,0.9261,0.7500,0.0262,48.41,0.3347
4,5,Decision Tree,Pohon Tunggal,0.9838,0.9262,0.9312,0.0049,0.9161,0.7083,0.0665,9.43,0.2524


AttributeError: Transformer log (type FunctionTransformer) does not provide get_feature_names_out.

In [2]:
# 3. Ekstraksi 10 Fitur Terpenting dari Extra Trees dan XGBoost (Perbaikan Nama Fitur)
def get_top_features(pipe, model_label, top_n=10):
    prep = pipe.named_steps["prep"]
    clf = pipe.named_steps["clf"]
    
    # Ambil nama fitur langsung dari masing-masing transformer
    cat_names = list(prep.named_transformers_["cat"].get_feature_names_out(["protocol_type", "service", "flag"]))
    log_names = ["duration", "src_bytes", "dst_bytes"]
    num_names = [c for c in SELECTED_FEATURES if c not in ["protocol_type", "service", "flag", "duration", "src_bytes", "dst_bytes"]]
    
    feat_names = cat_names + log_names + num_names
    importances = clf.feature_importances_
    
    df_fi = pd.DataFrame({
        "model": model_label,
        "feature": feat_names,
        "importance": importances,
    }).sort_values("importance", ascending=False).head(top_n)
    df_fi["rank"] = range(1, len(df_fi) + 1)
    return df_fi

et_pipe = joblib.load(models_dir / "extra_trees.joblib")
xgb_pipe = joblib.load(models_dir / "xgboost.joblib")

df_fi_all = pd.concat([
    get_top_features(et_pipe, "Extra Trees", 10),
    get_top_features(xgb_pipe, "XGBoost", 10),
], ignore_index=True)

df_fi_all.to_csv(results_dir / "feature_importance.csv", index=False)
print("\n=== TOP 10 FITUR TERPENTING (EXTRA TREES & XGBOOST) ===")
display(df_fi_all)

# 4. Analisis Kesalahan Khusus U2R dan R2L pada Model Peringkat 1
best_key = df_comp.loc[0, "model_key"]
best_name = df_comp.loc[0, "model_name"]
best_pipe = joblib.load(models_dir / f"{best_key}.joblib")
y_pred_best = best_pipe.predict(X_test)
cm_best = pd.DataFrame(
    confusion_matrix(y_test, y_pred_best, labels=URUTAN_KELAS),
    index=[f"Aktual_{c}" for c in URUTAN_KELAS],
    columns=[f"Pred_{c}" for c in URUTAN_KELAS],
)
print(f"\n=== MATRIKS KEKELIRUAN DETAIL R2L & U2R — {best_name} ===")
display(cm_best.loc[["Aktual_R2L", "Aktual_U2R"]])


=== TOP 10 FITUR TERPENTING (EXTRA TREES & XGBOOST) ===


,model,feature,importance,rank
0,Extra Trees,src_bytes,0.068665,1
1,Extra Trees,service_http,0.051131,2
2,Extra Trees,dst_host_srv_count,0.042621,3
3,Extra Trees,logged_in,0.041125,4
4,Extra Trees,dst_host_same_src_port_rate,0.038689,5
5,Extra Trees,dst_bytes,0.035655,6
6,Extra Trees,serror_rate,0.033082,7
7,Extra Trees,dst_host_diff_srv_rate,0.031837,8
8,Extra Trees,flag_SF,0.031742,9
9,Extra Trees,dst_host_same_srv_rate,0.031160,10



=== MATRIKS KEKELIRUAN DETAIL R2L & U2R — XGBoost ===


,Pred_Normal,Pred_DoS,Pred_Probe,Pred_R2L,Pred_U2R
Aktual_R2L,9,0,0,764,2
Aktual_U2R,3,0,0,1,20
